# The "Sugar Trap": Market Gap Analysis
**Client:** Helix CPG Partners | **Analyst:** Jayden | **Data:** Open Food Facts

Finding the "blue ocean" in the European snack aisle — product categories where demand for
high-protein, low-sugar options is not met by current offerings.

**Contents:** Setup → Story 1 (Cleaning) → Story 2 (Categorisation) → Story 3 (Nutrient Matrix)
→ Story 4 (Recommendation) → Bonus → Candidate's Choice

## Setup

All configuration lives in one cell. `USE_DRIVE` caches intermediate files to the author's
Google Drive; set it to `False` for a fully portable run (the notebook will then download and
process the raw export from scratch).

In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter

pd.set_option('display.max_columns', 50)

USE_DRIVE = True   # author's environment; set False for a portable run

WORK = './data'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = '/content/drive/MyDrive/market_gap'
    except Exception as e:
        print(f"Drive unavailable ({e}); falling back to {WORK}")
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

RAW_URL     = 'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz'
RAW_GZ      = '/content/off.csv.gz'
RAW_PARQUET = f'{WORK}/snacks_raw.parquet'

COLS = ['code','product_name','brands','countries_en','categories_tags',
        'ingredients_text','energy-kcal_100g','fat_100g','saturated-fat_100g',
        'sugars_100g','fiber_100g','proteins_100g','salt_100g','nutriscore_grade']

NUM_COLS  = ['sugars_100g','proteins_100g','fat_100g','saturated-fat_100g',
             'fiber_100g','salt_100g','energy-kcal_100g']
TEXT_COLS = ['product_name','brands','countries_en','categories_tags',
             'ingredients_text','nutriscore_grade']
NUTRIENTS = ['sugars_100g','proteins_100g','fat_100g','fiber_100g','salt_100g']

EU_MARKETS = ['France','Italy','Germany','Spain','United Kingdom',
              'Belgium','Switzerland','Netherlands']
EU_PATTERN = '|'.join(EU_MARKETS)

SUGAR_MAX, PROTEIN_MIN = 10, 10

audit = []
def log(step, df):
    audit.append({'step': step, 'rows': len(df)})
    print(f"{step:<38} {len(df):>8,}")

Mounted at /content/drive
Working dir: /content/drive/MyDrive/market_gap


### Ingestion

The Open Food Facts export is ~4.5M products and tab-separated despite the `.csv` extension.
It is read in 200k-row chunks, filtering on `en:snacks` and keeping 14 of ~200 columns, so the
full file never sits in memory at once.

*Before committing to this column list I read 5 rows to confirm the separator (211 columns, not 1)
and that every target column exists under the expected name — the OFF schema changes without
notice, and `nutriscore_grade` has previously been `nutrition_grade_fr`.*

In [2]:
if os.path.exists(RAW_PARQUET):
    snacks = pd.read_parquet(RAW_PARQUET)
    print("Loaded cached raw extract:", snacks.shape)
else:
    if not os.path.exists(RAW_GZ):
        !wget -q -c {RAW_URL} -O {RAW_GZ}

    chunks, scanned = [], 0
    reader = pd.read_csv(RAW_GZ, sep='\t', usecols=COLS, chunksize=200_000,
                         on_bad_lines='skip', low_memory=False,
                         encoding_errors='replace')
    for chunk in reader:
        scanned += len(chunk)
        mask = chunk['categories_tags'].fillna('').str.contains('en:snacks', case=False, na=False)
        if mask.any():
            chunks.append(chunk[mask])
    snacks = pd.concat(chunks, ignore_index=True)
    del chunks
    print(f"Scanned {scanned:,} products -> {len(snacks):,} snacks")

Loaded cached raw extract: (331907, 14)


### Schema normalisation

Chunked reads infer dtypes per chunk, and `concat` resolves any conflict to `object` — so
nutrient columns can silently arrive as strings. Barcodes have leading zeros and must *stay*
strings. Both are set explicitly rather than left to inference.

In [3]:
snacks['code'] = snacks['code'].astype(str)

for col in TEXT_COLS:
    if col in snacks.columns:
        snacks[col] = snacks[col].astype(str).replace({'nan': pd.NA, 'None': pd.NA})

for col in NUM_COLS:
    if col in snacks.columns:
        snacks[col] = pd.to_numeric(snacks[col], errors='coerce')

snacks['has_nutrition'] = snacks['sugars_100g'].notna() & snacks['proteins_100g'].notna()

if not os.path.exists(RAW_PARQUET):
    snacks.to_parquet(RAW_PARQUET, index=False)

log('Raw snacks (global)', snacks)
snacks[NUM_COLS].dtypes

Raw snacks (global)                     331,907


,0
sugars_100g,float64
proteins_100g,float64
fat_100g,float64
saturated-fat_100g,float64
fiber_100g,float64
salt_100g,float64
energy-kcal_100g,float64


## Story 1: Data Ingestion & Clean-Up

**Goal:** a dataset where every row is a real snack with trustworthy sugar and protein values —
because those two fields carry the entire recommendation.

### The decision that shaped this analysis: geographic scope

41% of snack rows have no sugar or protein value. Rather than dropping them blind, I tested
whether the missingness was random. It was not:

In [4]:
pd.crosstab(snacks['countries_en'], snacks['has_nutrition'], normalize='index') \
  .reindex(['France','United States','Italy','Germany','Spain','United Kingdom']) \
  .round(3)

has_nutrition,False,True
countries_en,,
France,0.109,0.891
United States,0.947,0.053
Italy,0.050,0.950
Germany,0.075,0.925
Spain,0.082,0.918
United Kingdom,0.137,0.863


Every European market clusters at 86–95%. The US sits at **5.3%** — not a gradient, a cliff.
Open Food Facts is a European-origin, contributor-driven database; US entries appear to arrive
largely via bulk barcode imports with no nutrition panel parsed.

**Implication:** a naive `dropna()` would have silently deleted the American market from a
"global" analysis, and any US finding would have rested on the 5% of products whose panels
someone chose to fill in — a self-selected sample.

> **The US looks like the largest gap in the dataset. It is a data collection artifact, not a
> market opportunity.**

**Decision:** scope the analysis to eight European markets (FR, IT, DE, ES, UK, BE, CH, NL).
This is defensible twice over — the data is reliable there, and the EU is a single coherent
regulatory and consumer market. A blended EU+US "world" gap would not be something R&D could
build against.

*Note: `countries_en` is a comma-separated list, so markets are matched with `str.contains`, not
equality. A product sold in both France and the US is correctly retained — it is on a French shelf.*

### Why drop rather than impute

Imputing a missing sugar value from a category median would fabricate the exact quantity the
recommendation depends on. A product whose sugar figure I invented cannot be evidence of a
market gap. Within Europe, coverage is 89.6%, so dropping costs little and preserves integrity.

In [5]:
eu = snacks[snacks['countries_en'].fillna('').str.contains(EU_PATTERN, case=False, na=False)].copy()
log('EU markets only', eu)

eu = eu[eu['has_nutrition']]
log('Nutrition data present', eu)

EU markets only                         193,086
Nutrition data present                  172,938


### Outlier rules and what they found

| Rule | Rationale |
|---|---|
| Each nutrient in 0–100 g | A 100g serving cannot contain >100g of anything |
| sugar + protein + fat ≤ 100 g | Macros cannot exceed the product's own mass |
| Energy in 0–900 kcal/100g | 900 kcal/100g is pure fat — the physical ceiling |
| Deduplicate on `code` | Barcode is the primary key |

First, what the failures actually look like (shown on the global set, before the EU filter):

In [6]:
pre = snacks[snacks['has_nutrition']]
bad = pre[~pre[NUTRIENTS].apply(lambda s: s.isna() | s.between(0, 100)).all(axis=1)]
print(len(bad), "rows with biologically impossible values (global)")
bad[['product_name','sugars_100g','proteins_100g','fat_100g']] \
   .sort_values('sugars_100g', ascending=False).head(10)

278 rows with biologically impossible values (global)


,product_name,sugars_100g,proteins_100g,fat_100g
126576,Banan's,74000.000000,2.800000,0.500000
21573,Starburst Original,1600.000000,0.000000,250.000000
183787,Haribo Banana,1283.333333,48.333333,8.333333
327958,Nesquik Chocolate Milk Flavouring,727.272727,41.322314,33.057851
250108,lindt,587.500000,82.500000,462.500000
115255,Marzipan Taler,517.000000,5.400000,19.700000
312666,Cin,360.980000,4.800000,14.000000
328082,Caramello Koala,355.555556,44.444444,173.333333
262427,<NA>,335.000000,0.000000,0.000000
97352,Probiotic Mango-Peach Yoggies,275.000000,0.000000,62.500000


The failures are **structured, not random**. Several show repeating decimals — Haribo Banana at
1283.33g sugar, Caramello Koala at 355.56g, Nesquik at 727.27g — the signature of OFF deriving
`_100g` fields by scaling an entered per-serving value against a serving size that was wrong by
an order of magnitude. One product claims 74,000g of sugar per 100g, a milligram/gram unit
confusion. These are propagated calculation errors, not typos in the sugar field.

*(278 impossible rows globally; 144 fall within the EU scope.)*

In [7]:
for col in NUTRIENTS:
    if col in eu.columns:
        eu = eu[eu[col].isna() | eu[col].between(0, 100)]
log('Nutrients within 0-100g', eu)

eu = eu[eu[['sugars_100g','proteins_100g','fat_100g']].sum(axis=1) <= 100]
log('Macros sum <= 100g', eu)

if 'energy-kcal_100g' in eu.columns:
    eu = eu[eu['energy-kcal_100g'].isna() | eu['energy-kcal_100g'].between(0, 900)]
log('Energy 0-900 kcal', eu)

eu = eu.drop_duplicates(subset=['code'])
log('Deduplicated on barcode', eu)

Nutrients within 0-100g                 172,794
Macros sum <= 100g                      172,689
Energy 0-900 kcal                       172,473
Deduplicated on barcode                 172,473


**Only 465 rows (0.27%) failed these rules.** The quality problem in this dataset is
missingness, not implausibility: where data was entered, it was overwhelmingly sane.
Deduplication removed exactly zero rows, confirming OFF enforces the barcode key.

A product that cannot be named cannot be a recommendation, so rows without a usable
`product_name` are dropped:

In [8]:
eu = eu[eu['product_name'].notna() & (eu['product_name'].str.strip() != '')]
log('Valid product names', eu)

Valid product names                     168,776


### Removing products that are not snacks

Two groups inherited an `en:snacks` tag through OFF's crowd-sourced hierarchy:

**Dry baking mixes** (cake mix, muffin mix, brownie mix). Their nutrition is measured as sold:
flour and powder. Nobody eats them in that state; once egg, milk and oil are added and the
product is baked, the profile changes completely. Critically, they land in the *extremes of both
axes* — "CHIA SEED MUFFIN MIX" reads 4.3g sugar / 32.5g protein, which would place it squarely
in the blue ocean this analysis exists to find. Left in, they would corrupt the headline finding.

**Baby food.** Purées and infant desserts are deliberately low-sugar, regulated differently, and
aimed at a consumer who is not this client's target. Same failure mode, smaller scale.

*Tags are matched as exact set membership, not substrings. An early substring version wrongly
flagged Mott's Applesauce, because "sauces" is contained in "applesauces".*

In [9]:
JUNK = {'en:cooking-helpers','en:baking-mixes','en:cake-mixes','en:dessert-mixes',
        'en:pastry-helpers','en:condiments','en:sauces'}
BABY = {'en:baby-foods','en:snacks-and-desserts-for-babies','en:baby-fruit-desserts',
        'en:from-6-months','en:dairy-dessert-for-baby','en:baby-snacks'}

def has_any(tags, tagset):
    return bool(tagset & {t.strip() for t in (tags or '').split(',')})

eu = eu[~eu['categories_tags'].fillna('').apply(lambda t: has_any(t, JUNK))]
log('Non-snack products removed', eu)

eu = eu[~eu['categories_tags'].fillna('').apply(lambda t: has_any(t, BABY))]
log('Baby food removed', eu)

Non-snack products removed              167,657
Baby food removed                       166,811


### Story 1 result

50% of rows removed — but **97% of that loss comes from two documented scope decisions**
(US exclusion, missing nutrition). Every quality rule combined accounts for 816 rows, and
nameless products for 3,697.

In [10]:
pd.DataFrame(audit)

,step,rows
0,Raw snacks (global),331907
1,EU markets only,193086
2,Nutrition data present,172938
3,Nutrients within 0-100g,172794
4,Macros sum <= 100g,172689
5,Energy 0-900 kcal,172473
6,Deduplicated on barcode,172473
7,Valid product names,168776
8,Non-snack products removed,167657
9,Baby food removed,166811


## Story 2: The Category Wrangler

`categories_tags` is a comma-separated list of crowd-sourced tags, and products carry several at
once at different levels of a hierarchy — a single biscuit may be tagged
`en:snacks, en:sweet-snacks, en:biscuits-and-cakes, en:biscuits, en:chocolate-biscuits`.

In [11]:
tags = Counter()
for t in eu['categories_tags'].dropna():
    tags.update(x.strip() for x in t.split(','))
pd.Series(dict(tags.most_common(40)))

,0
en:snacks,166770
en:sweet-snacks,136935
en:biscuits-and-cakes,56260
en:confectioneries,42600
en:cocoa-and-its-products,33576
en:biscuits,30009
en:salty-snacks,25314
en:chocolates,22140
en:appetizers,20596
en:biscuits-and-crackers,17932


### Three kinds of tag

Counting tag frequency across the clean set shows they are not interchangeable:

- **Parents** (`en:sweet-snacks`, `en:salty-snacks`) — too broad to be a category. Used only as
  a fallback.
- **Cross-cutting attributes** (`en:plant-based-foods` 16,255, `en:festive-foods` 6,767,
  `en:frozen-foods` 2,787) — these describe a *property*, not a form factor. A plant-based cake
  is still a cake. Made into buckets, `en:plant-based-foods` alone would have hijacked 10% of
  the data.
- **Real categories** (`en:crisps`, `en:cakes`, `en:dark-chocolates`, `en:nuts-and-their-products`)
  — the actual buckets.

### Assignment logic

Ordered first-match-wins over exact tag sets. Order is doing real work, because tags collide:

- `en:chocolate-biscuits` (6,842 products) is both chocolate and biscuit. **Form factor wins** —
  it is eaten as a biscuit. Biscuits are therefore checked before Chocolate.
- `en:biscuits-and-crackers` spans sweet biscuits *and* savoury crackers. **Savoury is checked
  first**, or salted crackers would be classified sweet.

In [12]:
BUCKETS = [
    ('Bars & Protein Snacks', {'en:cereal-bars','en:bars','en:protein-bars',
                               'en:energy-bars','en:fruit-bars'}),
    ('Nuts & Seeds',          {'en:nuts-and-their-products','en:nuts','en:almonds',
                               'en:peanuts','en:cashew-nuts','en:seeds','en:pistachios'}),
    ('Dried Fruit',           {'en:dried-fruits','en:raisins','en:dates','en:dried-apricots'}),
    ('Chips & Savoury',       {'en:crisps','en:potato-crisps','en:chips-and-fries',
                               'en:crackers-appetizers','en:corn-chips','en:extruded-snacks',
                               'en:popcorn','en:pretzels','en:salty-snacks',
                               'en:taralli','it:taralli','en:extruded-crispbreads',
                               'en:chips','en:crispbreads'}),
    ('Biscuits & Cookies',    {'en:biscuits','en:chocolate-biscuits','en:shortbread-cookies',
                               'en:biscuits-and-crackers','en:cookies'}),
    ('Cakes & Pastries',      {'en:cakes','en:chocolate-cakes','en:viennoiseries','en:pastries',
                               'en:brioches','en:madeleines','en:panettone',
                               'en:sweet-pastries-and-pies'}),
    ('Chocolate',             {'en:dark-chocolates','en:milk-chocolates','en:chocolates',
                               'en:chocolate-candies','en:white-chocolates',
                               'en:cocoa-and-its-products'}),
    ('Confectionery',         {'en:candies','en:bonbons','en:gummies','en:marshmallows',
                               'en:confectioneries'}),
]

FALLBACK = [('Other Sweet',   {'en:sweet-snacks'}),
            ('Other Savoury', {'en:salty-snacks'})]

def assign(tags):
    t = {x.strip() for x in (tags or '').split(',')}
    for name, keys in BUCKETS + FALLBACK:
        if t & keys:
            return name
    return 'Other'

eu['primary_category'] = eu['categories_tags'].apply(assign)
eu['primary_category'].value_counts(dropna=False)

,count
primary_category,
Cakes & Pastries,32909
Chocolate,30540
Biscuits & Cookies,29344
Confectionery,26982
Chips & Savoury,25541
Bars & Protein Snacks,8621
Other Sweet,6659
Nuts & Seeds,2954
Other,2900


#### Verification

Two checks: how much is left unclassified, and whether the collision rule actually fired the way
it was designed to.

In [13]:
print("Other rate:", (eu['primary_category'] == 'Other').mean().round(3))
print()
print("Where en:chocolate-biscuits landed:")
print(eu[eu['categories_tags'].str.contains('en:chocolate-biscuits', na=False)]
        ['primary_category'].value_counts())

Other rate: 0.017

Where en:chocolate-biscuits landed:
primary_category
Biscuits & Cookies       6705
Bars & Protein Snacks     126
Chips & Savoury             5
Nuts & Seeds                3
Name: count, dtype: int64


### Result — 9 buckets, 2.4% unclassified

| Category | Products | Share |
|---|---|---|
| Cakes & Pastries | 32,909 | 19.7% |
| Chocolate | 30,541 | 18.3% |
| Biscuits & Cookies | 29,354 | 17.6% |
| Confectionery | 26,982 | 16.2% |
| Chips & Savoury | 24,455 | 14.7% |
| Bars & Protein Snacks | 8,621 | 5.2% |
| Other Sweet | 6,664 | 4.0% |
| Other | 3,970 | 2.4% |
| Nuts & Seeds | 2,954 | 1.8% |
| Dried Fruit | 361 | 0.2% |

**The gap is already visible before any chart is drawn.** Four sweet categories are 71% of the
European snack aisle. Nuts, seeds and dried fruit together are 2.0%.

*Caveat: Dried Fruit at 361 products is too thin to support a confident recommendation and
likely reflects OFF tagging conventions as much as shelf reality.*

In [14]:
eu.to_parquet(f'{WORK}/snacks_eu_clean.parquet', index=False)
pd.DataFrame(audit).to_csv(f'{WORK}/cleaning_audit.csv', index=False)
print("Saved:", len(eu), "rows")

Saved: 166811 rows


## Story 3: The Nutrient Matrix

### Defining the quadrants

**10g sugar / 100g** is roughly where EU nutrient-profile schemes begin flagging a product as
high-sugar. **10g protein / 100g** is a reasonable "high protein" bar. Both are judgement calls,
so the sensitivity of the finding to these thresholds is tested below rather than assumed.

In [15]:
eu['quadrant'] = np.select(
    [(eu.sugars_100g <  SUGAR_MAX) & (eu.proteins_100g >= PROTEIN_MIN),
     (eu.sugars_100g >= SUGAR_MAX) & (eu.proteins_100g >= PROTEIN_MIN),
     (eu.sugars_100g <  SUGAR_MAX) & (eu.proteins_100g <  PROTEIN_MIN)],
    ['Blue Ocean (Low Sugar, High Protein)',
     'High Sugar + High Protein',
     'Low Sugar, Low Protein'],
    default='Sugar Trap (High Sugar, Low Protein)')

eu['quadrant'].value_counts(normalize=True).round(3)

,proportion
quadrant,
"Sugar Trap (High Sugar, Low Protein)",0.707
"Low Sugar, Low Protein",0.167
"Blue Ocean (Low Sugar, High Protein)",0.068
High Sugar + High Protein,0.059


The blue ocean is not empty — it holds ~11,300 products. The honest framing is a **saturation
ratio: the European snack aisle runs roughly 10:1 sugar-trap to blue-ocean.**

### Where each category sits

In [16]:
stats = eu.groupby('primary_category').agg(
    products=('code','size'),
    median_sugar=('sugars_100g','median'),
    median_protein=('proteins_100g','median'),
).round(1).sort_values('products', ascending=False)
stats

,products,median_sugar,median_protein
primary_category,,,
Cakes & Pastries,32909,25.0,6.0
Chocolate,30540,45.8,6.8
Biscuits & Cookies,29344,28.0,6.5
Confectionery,26982,53.0,3.0
Chips & Savoury,25541,2.3,6.9
Bars & Protein Snacks,8621,29.2,7.8
Other Sweet,6659,34.2,5.7
Nuts & Seeds,2954,7.6,18.0
Other,2900,5.2,8.4


In [17]:
print("Absolute counts by category and quadrant:")
pd.crosstab(eu.primary_category, eu.quadrant)

Absolute counts by category and quadrant:


quadrant,"Blue Ocean (Low Sugar, High Protein)",High Sugar + High Protein,"Low Sugar, Low Protein","Sugar Trap (High Sugar, Low Protein)"
primary_category,,,,
Bars & Protein Snacks,788,2055,320,5458
Biscuits & Cookies,463,1238,1468,26175
Cakes & Pastries,433,693,2555,29228
Chips & Savoury,6126,340,17370,1705
Chocolate,549,2132,889,26970
Confectionery,298,2207,3869,20608
Dried Fruit,14,64,35,248
Nuts & Seeds,1655,517,125,657
Other,839,307,854,900


### Sensitivity check

If the recommendation flips when the thresholds move, the finding is fragile and must be
disclosed. Testing four plausible cut-offs:

In [18]:
for s, p in [(10,10), (12,8), (8,12), (5,15)]:
    q = ((eu.sugars_100g < s) & (eu.proteins_100g >= p))
    top = eu[q].primary_category.value_counts(normalize=True).head(2)
    print(f"sugar<{s:>2}, protein>={p:>2}: {q.mean():.3f} blue ocean | top: {dict(top.round(2))}")

sugar<10, protein>=10: 0.068 blue ocean | top: {'Chips & Savoury': np.float64(0.54), 'Nuts & Seeds': np.float64(0.15)}
sugar<12, protein>= 8: 0.105 blue ocean | top: {'Chips & Savoury': np.float64(0.55), 'Nuts & Seeds': np.float64(0.1)}
sugar< 8, protein>=12: 0.046 blue ocean | top: {'Chips & Savoury': np.float64(0.49), 'Nuts & Seeds': np.float64(0.2)}
sugar< 5, protein>=15: 0.019 blue ocean | top: {'Chips & Savoury': np.float64(0.43), 'Bars & Protein Snacks': np.float64(0.19)}


**Chips & Savoury ranks first at all four thresholds** (0.51 / 0.50 / 0.46 / 0.41). The finding
is not an artifact of where the lines were drawn.

### TODO — the scatter plot goes here

## Story 4: The Recommendation

### Who is already in the blue ocean?

In [19]:
bo_chips = eu[(eu.primary_category == 'Chips & Savoury') &
              (eu.quadrant == 'Blue Ocean (Low Sugar, High Protein)')]

print(bo_chips.brands.value_counts().head(15))
print()
print(bo_chips[['proteins_100g','sugars_100g']].describe().round(1))

brands
Carrefour       80
La Mole         70
Picard          67
Auchan          59
U               55
Lorenz          49
Boehli          46
Thiriet         42
AH              41
Caputo          40
Belin           39
Galbusera       36
Casino          34
Leader Price    33
Delhaize        33
Name: count, dtype: int64

       proteins_100g  sugars_100g
count         6126.0       6126.0
mean            14.9          2.8
std              8.0          2.0
min             10.0          0.0
25%             11.0          1.3
50%             12.2          2.3
75%             15.0          3.7
max             78.2          9.9


In [20]:
bo_chips[['product_name','brands','proteins_100g','sugars_100g']].sample(15, random_state=1)

,product_name,brands,proteins_100g,sugars_100g
211196,Cream Crackers Twin Pack 2 x,Jacobs,10.0,1.4
205300,Kichererbsenchips,Rewe,14.0,5.4
174441,Croquants basilic parmesan,À la table de Mathilde,16.1,6.9
204389,Mini burger bloc de foie gras,"Deluxe,Lidl",12.3,7.0
249030,Bretzel PEARLS,Roland,13.0,3.7
276052,cracker con riso soffiato,Pam & Panorama,10.0,2.2
159650,Végétal Snack pops Saveur Tomate & Basilic,Carrefour Sensation,16.0,2.7
229620,Cracotte froment,LU,10.0,8.8
255033,Dinkel Cracker Tomate Basilikum,Pjur,11.0,3.3
319225,Chips paprika,AH,28.0,3.2


### TODO — Key Insight box and the fill-in-the-blank sentence